# Official motivation tail scaling
Tail token behavior uses the same official task records as the main evaluation.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
frames = []
for backend in ('deepseek_harness', 'codex'):
    p = RESULTS / backend / 'official_token_summary.csv'
    if p.exists(): frames.append(pd.read_csv(p))
df = pd.concat(frames, ignore_index=True).drop_duplicates() if frames else pd.DataFrame()
if not df.empty:
    order = {'short': 1, 'medium': 2, 'long': 3}
    df['scale_id'] = df['scale'].map(order)
    view = df.groupby(['harness_backend', 'suite', 'mode', 'scale_id'], as_index=False)[['total_tokens_p95', 'total_tokens_p99']].mean()
    fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.8), dpi=300, sharey=True)
    for ax, metric, title in zip(axes, ['total_tokens_p95', 'total_tokens_p99'], ['p95 tokens', 'p99 tokens']):
        for (backend, suite, mode), group in view.groupby(['harness_backend', 'suite', 'mode']):
            group = group.sort_values('scale_id')
            ax.plot(group['scale_id'], group[metric], marker='o', linewidth=1.0, label=f'{backend}:{suite}:{mode}')
        ax.set_xticks([1, 2, 3], ['short', 'medium', 'long'])
        ax.set_xlabel('Trajectory length (# calls)', fontsize=8)
        ax.set_title(title, fontsize=8)
    axes[0].set_ylabel('Tokens', fontsize=8)
    axes[1].legend(fontsize=5.5, frameon=True)
    fig.tight_layout()
    fig.savefig(ROOT / 'motivation' / 'FIG-Motivation-Official-Tail.pdf', bbox_inches='tight')
else:
    print('No official summary found.')
